In [ ]:
import polars as pl
from ete4 import NCBITaxa, Tree
from pathlib import Path

# entrez
from Bio import Entrez
import dotenv
import os


def validade_tree(tree, taxa):
    """Check if the taxa in the tree match the provided taxa list."""
    tree_taxa = set(leaf.name for leaf in tree.leaves())
    missing_in_tree = taxa - tree_taxa
    extra_in_tree = tree_taxa - taxa
    if missing_in_tree:
        print(f"Taxa missing in tree: {', '.join(missing_in_tree)}")
    if extra_in_tree:
        print(f"Extra taxa in tree: {', '.join(extra_in_tree)}")
    return missing_in_tree, extra_in_tree

In [ ]:
dotenv.load_dotenv()

Entrez.email = os.getenv("ENTREZ_EMAIL")
Entrez.api_key = os.getenv("ENTREZ_API_KEY")

In [ ]:
# Assuming either all *_compressed.tar.zst files or all_groups_lean.tar.zst file is/are decompressed
CWD = Path().resolve()
BASE = CWD.parent

print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
groups = [
    "fungi_mit",
    "metazoans_mit",
    "plants_mit",
    "plants_plt",
    "protists_mit",
    "protists_plt",
    "green_algae_mit",
    "green_algae_plt",
]

In [ ]:
ncbi_taxa = NCBITaxa()
ncbi_taxa.update_taxonomy_database()
# July 7th 2026, 14h EDT

In [ ]:
all_data = pl.DataFrame()
for g in groups:
    gdir = BASE / g

    tsv = pl.read_csv(gdir / f"{g}.tsv", separator="\t")
    ans = tsv["AN"].n_unique()
    taxids = tsv["ncbi_taxid"].n_unique()

    print(f"Validating tree for group: {g} | ans: {ans} | taxids: {taxids}")

    tree = ncbi_taxa.get_topology(tsv["ncbi_taxid"])
    tree.to_ultrametric()

    print("Tree created, now validating it...")
    taxa_in_tsv = set(tsv["ncbi_taxid"].cast(pl.String))
    missing_in_tree, extra_in_tree = validade_tree(tree, taxa_in_tsv)

    if missing_in_tree:
        missing_taxa = [int(g) for g in missing_in_tree]
        ans_missing = tsv.filter(pl.col("ncbi_taxid").is_in(missing_taxa)).select(
            ["AN", "Species", "ncbi_taxid"]
        )

    print(f"\t len missing_in_tree: {len(missing_in_tree)}")
    print(f"\t len ans_missing: {ans_missing["AN"].n_unique()}")
    print(f"\t len extra_in_tree. : {len(extra_in_tree)}")

    tsv = tsv.with_columns(pl.lit(g).alias("group"))
    all_data = pl.concat([all_data, tsv], how="vertical")

    if not missing_in_tree:
        tree.write(outfile=gdir / "tree.nwk", parser=1)

    print()

In [ ]:
# all tree creation
print("Creating the all tree...")
all_tree = ncbi_taxa.get_topology(all_data["ncbi_taxid"])
missing_in_tree, extra_in_tree = validade_tree(
    all_tree, set(all_data["ncbi_taxid"].cast(pl.String))
)

if missing_in_tree:
    missing_taxa = [int(g) for g in missing_in_tree]
    ans_missing = all_data.filter(pl.col("ncbi_taxid").is_in(missing_taxa)).select(
        ["AN", "Species", "ncbi_taxid"]
    )

print(f"\t len missing_in_tree: {len(missing_in_tree)}")
print(f"\t len ans_missing: {ans_missing["AN"].n_unique()}")
print(f"\t len extra_in_tree. : {len(extra_in_tree)}")

if not missing_in_tree:
    all_tree.write(outfile=BASE / "all_tree.nwk", parser=1)